# Cricket Fantasy Agent System

## Phase 1: Configuration & Shared Setup

**Purpose**: Initialize LLM, API credentials, and shared utilities for caching and API calls.

**Key Components**:
- LangChain LLM initialization (Groq)
- Shared HEADERS for all Cricbuzz API calls
- Cache management functions (`load_cache()`, `save_cache()`)
- Single source of truth for API configuration

In [ ]:
# ==========================================
# PHASE 1: IMPORTS WITH COMPREHENSIVE ERROR HANDLING
# ==========================================

import os
import json
import logging
from pathlib import Path
from typing import Optional

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables with error handling
try:
    from dotenv import load_dotenv
    load_dotenv()
    logger.info("✅ Environment variables loaded from .env file")
except ImportError as e:
    logger.warning(f"⚠️ python-dotenv not installed: {e}. Using fallback environment.")
except Exception as e:
    logger.error(f"❌ Failed to load .env file: {e}")

# Initialize environment variables with fallback values
env_vars = {
    'GOOGLE_API_KEY': 'GOOGLE_API_KEY',
    'TAVILY_API_KEY': 'TAVILY_API_KEY',
    'GROQ_API_KEY': 'GROQ_API_KEY',
    'RAPIDAPI_KEY': 'RAPIDAPI_KEY',
    'WEATHER_API_KEY': 'WEATHER_API_KEY'
}

for var_name, var_key in env_vars.items():
    try:
        value = os.getenv(var_key)
        if value:
            os.environ[var_name] = value
            logger.info(f"✅ {var_name} loaded successfully")
        else:
            logger.warning(f"⚠️ {var_name} not found in environment. Using default.")
    except Exception as e:
        logger.error(f"❌ Error loading {var_name}: {e}")

# Initialize LangChain chat model with error handling
try:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(model='openai/gpt-oss-120b', model_provider="groq")
    logger.info("✅ LangChain model initialized successfully (Groq)")
except ImportError as e:
    logger.error(f"❌ Failed to import init_chat_model: {e}")
    raise SystemExit("Required dependency 'langchain' not found. Please install it.")
except Exception as e:
    logger.error(f"❌ Failed to initialize chat model: {e}")
    raise SystemExit(f"Chat model initialization failed: {e}")
# model = init_chat_model(model='gemini-3-pro-preview',model_provider="google_genai",base_url="https://api.ai-gateway.tigeranalytics.com")

# ==========================================
# SHARED CONFIGURATION (For All APIs & Caching)
# ==========================================
from datetime import datetime, timedelta

# API Headers - Load from .env for security
HEADERS = {
    "x-rapidapi-key": os.getenv('RAPIDAPI_KEY', ''),
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

CACHE_FILE = "cricket_cache.json"

# ==========================================
# CACHE CLASS - Optimized Cache Management with Error Handling
# ==========================================
class CacheManager:
    """Efficient cache manager with comprehensive error handling and cleanup mechanism."""

    def __init__(self, cache_file: str):
        self.cache_file = cache_file
        self.cache_data = self._load()
        logger.info(f"✅ CacheManager initialized with {self.size()} cached entries")

    def _load(self) -> dict:
        """Load cache from file (called once on init)."""
        if not os.path.exists(self.cache_file):
            logger.info(f"ℹ️ Cache file '{self.cache_file}' not found. Starting fresh.")
            return {}

        try:
            with open(self.cache_file, "r") as f:
                data = json.load(f)
                logger.info(f"✅ Cache loaded: {len(data)} entries")
                return data
        except json.JSONDecodeError as e:
            logger.error(f"❌ Cache file corrupted (JSON decode error): {e}. Starting fresh.")
            return {}
        except IOError as e:
            logger.error(f"❌ Failed to read cache file: {e}. Starting fresh.")
            return {}
        except Exception as e:
            logger.error(f"❌ Unexpected error loading cache: {e}. Starting fresh.")
            return {}

    def _save(self) -> None:
        """Save cache to file with error handling."""
        temp_file = f"{self.cache_file}.tmp"
        try:
            # Write to temp file first (atomic write pattern)
            with open(temp_file, "w") as f:
                json.dump(self.cache_data, f, indent=4)
            # Rename temp file to actual file
            os.replace(temp_file, self.cache_file)
            logger.debug(f"✅ Cache saved successfully ({self.size()} entries)")
        except IOError as e:
            logger.error(f"❌ Failed to save cache: {e}")
            # Clean up temp file if it exists
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass
        except Exception as e:
            logger.error(f"❌ Unexpected error saving cache: {e}")
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass

    def get(self, key: str) -> Optional[dict]:
        """Get value from cache. Returns None if not found."""
        try:
            return self.cache_data.get(key)
        except Exception as e:
            logger.warning(f"⚠️ Error retrieving cache key '{key}': {e}")
            return None

    def set(self, key: str, value) -> None:
        """Set value in cache with timestamp. Auto-saves to file."""
        try:
            self.cache_data[key] = {
                "data": value,
                "timestamp": datetime.now().isoformat()
            }
            self._save()
        except TypeError as e:
            logger.error(f"❌ Failed to serialize cache value for key '{key}': {e}")
        except Exception as e:
            logger.error(f"❌ Unexpected error setting cache key '{key}': {e}")

    def exists(self, key: str) -> bool:
        """Check if key exists in cache."""
        try:
            return key in self.cache_data
        except Exception as e:
            logger.warning(f"⚠️ Error checking cache key '{key}': {e}")
            return False

    def cleanup(self, days: int = 7) -> int:
        """Remove cache entries older than X days. Returns count removed."""
        keys_to_remove = []
        try:
            cutoff_date = datetime.now() - timedelta(days=days)

            for key, entry in self.cache_data.items():
                try:
                    if isinstance(entry, dict) and "timestamp" in entry:
                        entry_date = datetime.fromisoformat(entry["timestamp"])
                        if entry_date < cutoff_date:
                            keys_to_remove.append(key)
                except ValueError as e:
                    logger.warning(f"⚠️ Invalid timestamp format for key '{key}': {e}. Marking for removal.")
                    keys_to_remove.append(key)
                except Exception as e:
                    logger.warning(f"⚠️ Error processing entry '{key}': {e}")

            # Remove expired entries
            for key in keys_to_remove:
                try:
                    del self.cache_data[key]
                except KeyError:
                    pass

            # Save if any entries were removed
            if keys_to_remove:
                self._save()
                logger.info(f"🧹 Cleanup: Removed {len(keys_to_remove)} expired cache entries (older than {days} days)")
            else:
                logger.info(f"ℹ️ Cleanup: No expired entries found (threshold: {days} days)")

            return len(keys_to_remove)
        except Exception as e:
            logger.error(f"❌ Unexpected error during cleanup: {e}")
            return 0

    def clear(self) -> None:
        """Clear all cache entries."""
        try:
            old_size = self.size()
            self.cache_data = {}
            self._save()
            logger.info(f"🗑️ Cache cleared ({old_size} entries removed)")
        except Exception as e:
            logger.error(f"❌ Failed to clear cache: {e}")

    def size(self) -> int:
        """Get number of cache entries."""
        try:
            return len(self.cache_data)
        except Exception as e:
            logger.warning(f"⚠️ Error getting cache size: {e}")
            return 0

# Initialize global cache instance (used by all tools)
try:
    cache = CacheManager(CACHE_FILE)
    logger.info("✅ Cache system initialized successfully")
except Exception as e:
    logger.error(f"❌ Failed to initialize cache system: {e}")
    # Create a fallback in-memory cache
    class MemoryCacheManager:
        def __init__(self):
            self.cache_data = {}
        def exists(self, key): return key in self.cache_data
        def get(self, key): return self.cache_data.get(key)
        def set(self, key, value): self.cache_data[key] = {"data": value, "timestamp": datetime.now().isoformat()}
        def cleanup(self, days=7): return 0
        def clear(self): self.cache_data = {}
        def size(self): return len(self.cache_data)

    cache = MemoryCacheManager()
    logger.warning("⚠️ Using fallback in-memory cache (file cache unavailable)")

2026-02-26 13:21:17,945 - INFO - ✅ Environment variables loaded from .env file
2026-02-26 13:21:17,946 - INFO - ✅ GOOGLE_API_KEY loaded successfully
2026-02-26 13:21:17,948 - INFO - ✅ TAVILY_API_KEY loaded successfully
2026-02-26 13:21:17,949 - INFO - ✅ GROQ_API_KEY loaded successfully
2026-02-26 13:21:17,951 - INFO - ✅ RAPIDAPI_KEY loaded successfully
2026-02-26 13:21:17,952 - INFO - ✅ WEATHER_API_KEY loaded successfully
2026-02-26 13:21:29,953 - INFO - ✅ LangChain model initialized successfully (Groq)
2026-02-26 13:21:29,962 - INFO - ✅ Cache loaded: 1 entries
2026-02-26 13:21:29,963 - INFO - ✅ CacheManager initialized with 1 cached entries
2026-02-26 13:21:29,964 - INFO - ✅ Cache system initialized successfully


## Phase 2: Tool Definitions (Cached)

**Purpose**: Define all LangChain tools with built-in caching to avoid redundant API calls.

**Tools Defined**:
- `get__weathr()` - Fetch weather for a specific city/date (Visual Crossing API)
- `get_match_venue()` - Fetch stadium info & pitch history (Tavily Search + caching)
- `get_match_environment()` - Fetch current pitch report (Tavily Search + caching)
- `get_cricket_squads_by_match()` - Fetch playing XI for both teams (Cricbuzz API + caching)

**Cache Strategy**: All tools cache their responses using shared `cricket_cache.json` file

In [ ]:
# ==========================================
# PHASE 2: TOOL IMPORTS WITH ERROR HANDLING
# ==========================================

# Import tools with comprehensive error handling
try:
    from langchain.tools import tool
    logger.info("✅ LangChain tools imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import LangChain tools: {e}")
    raise SystemExit("Required dependency 'langchain' not found.")

try:
    from langchain_tavily import TavilySearch
    logger.info("✅ Tavily Search imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import Tavily Search: {e}")
    raise SystemExit("Required dependency 'langchain-tavily' not found.")

try:
    import requests
    logger.info("✅ Requests library imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import requests: {e}")
    raise SystemExit("Required dependency 'requests' not found.")

# Initialize Tavily client with error handling
try:
    tavily_client = TavilySearch()
    logger.info("✅ Tavily Search client initialized successfully")
except Exception as e:
    logger.error(f"❌ Failed to initialize Tavily client: {e}")
    tavily_client = None

# ==========================================
# CACHED TOOL DEFINITIONS (ALL TOOLS - Used by Both Agents)
# ==========================================

@tool
def get__weathr(city: str, date: str) -> str:
    """
    Fetches the weather for a specific city on a specific date with caching.
    Input 'city' should be a string (e.g., 'Chennai').
    Input 'date' should be in YYYY-MM-DD format.
    """
    cache_key = f"weather_{city}_{date}"

    try:
        # Check cache first (no file I/O)
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Weather cache hit for {city} on {date}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached weather: {e}")

        API_KEY = os.getenv('WEATHER_API_KEY')
        if not API_KEY:
            logger.error("❌ WEATHER_API_KEY not configured in environment")
            return "Error: Weather API key not configured. Please set WEATHER_API_KEY in .env"

        url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/{city}/{date}?unitGroup=metric&key={API_KEY}&contentType=json"

        response = requests.get(url, timeout=10)

        if response.status_code == 401:
            logger.error("❌ Weather API authentication failed (401)")
            return "Error: Weather API authentication failed. Check WEATHER_API_KEY."
        elif response.status_code == 404:
            logger.warning(f"⚠️ Weather data not found for {city} on {date}")
            return f"Weather data not available for {city} on {date}"
        elif response.status_code != 200:
            logger.error(f"❌ Weather API returned status {response.status_code}")
            return f"Error: API returned status code {response.status_code}"

        data = response.json()
        day_data = data.get('days', [{}])[0]
        temp = day_data.get('temp')
        conditions = day_data.get('conditions')
        description = day_data.get('description', 'No description available.')

        result = (f"Weather in {city} on {date}: "
                  f"Temperature: {temp}°C, "
                  f"Conditions: {conditions}. "
                  f"Summary: {description}")

        # Store in cache (single write operation)
        cache.set(cache_key, result)
        logger.info(f"✅ Weather for {city} retrieved and cached")
        return result

    except requests.Timeout:
        logger.error(f"❌ Weather API request timed out for {city}/{date}")
        return f"Error: Weather API request timed out (10s timeout)"
    except requests.ConnectionError as e:
        logger.error(f"❌ Connection error fetching weather: {e}")
        return f"Error: Connection error to weather service: {str(e)}"
    except requests.RequestException as e:
        logger.error(f"❌ Weather API request failed: {e}")
        return f"Error: Failed to fetch weather data: {str(e)}"
    except (KeyError, IndexError, ValueError) as e:
        logger.error(f"❌ Failed to parse weather API response: {e}")
        return f"Error: Invalid weather API response format"
    except Exception as e:
        logger.error(f"❌ Unexpected error in get__weathr: {e}")
        return f"Error: Unexpected error fetching weather: {str(e)}"

@tool
def get_match_venue(match: str) -> str:
    """
    REQUIRED: Fetches the official stadium name, city, and critical venue history with caching.
    Includes average first innings scores and pitch history.
    """
    cache_key = f"venue_{match.lower().replace(' ', '_')}"

    try:
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Venue cache hit for {match}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached venue: {e}")

        if tavily_client is None:
            logger.error("❌ Tavily client not initialized")
            return "Error: Tavily search service not available"

        query = (f"official venue for {match} today, average first innings score men and women T20, "
                 f"is it a batting or bowling pitch, stadium history and record")

        result = str(tavily_client.invoke(query))
        cache.set(cache_key, result)
        logger.info(f"✅ Venue information retrieved and cached for {match}")
        return result

    except Exception as e:
        logger.error(f"❌ Error fetching venue for {match}: {e}")
        return f"Error: Failed to fetch venue information: {str(e)}"

@tool
def get_match_environment(location: str) -> str:
    """
    Fetches pitch report status for a specific stadium with caching.
    Use this to determine if the conditions favor bowlers or batsmen.
    """
    cache_key = f"pitch_{location.lower().replace(' ', '_')}"

    try:
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Pitch cache hit for {location}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached pitch info: {e}")

        if tavily_client is None:
            logger.error("❌ Tavily client not initialized")
            return "Error: Tavily search service not available"

        query = f"current pitch report for cricket match at {location}"
        result = str(tavily_client.invoke(query))
        cache.set(cache_key, result)
        logger.info(f"✅ Pitch report retrieved and cached for {location}")
        return result

    except Exception as e:
        logger.error(f"❌ Error fetching pitch environment for {location}: {e}")
        return f"Error: Failed to fetch pitch environment: {str(e)}"

@tool
def get_cricket_squads_by_match(match_query: str) -> str:
    """
    Retrieves the playing 11 and player IDs for both teams with caching.
    Input should be the two teams, e.g., 'MI vs GG Women'.
    Required to get player_ids for batch stats retrieval.
    """
    url_live = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"
    cache_key = f"squad_{match_query.lower().replace(' ', '_')}"

    try:
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Squad cache hit for {match_query}")
                return f"[CACHED SQUAD] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached squad: {e}")

        if not HEADERS.get('x-rapidapi-key'):
            logger.error("❌ RapidAPI key not configured")
            return "Error: RapidAPI key not configured. Please set RAPIDAPI_KEY in .env"

        response = requests.get(url_live, headers=HEADERS, timeout=10)

        if response.status_code == 403:
            logger.error("❌ Cricket API access forbidden (403)")
            return "Error: API access forbidden. Check RAPIDAPI_KEY permissions."
        elif response.status_code == 401:
            logger.error("❌ Cricket API authentication failed (401)")
            return "Error: API authentication failed. Check RAPIDAPI_KEY."
        elif response.status_code != 200:
            logger.error(f"❌ Cricket API returned status {response.status_code}")
            return f"Error: API returned status code {response.status_code}"

        data = response.json()
        logger.debug("✅ Response received for cricket matches search")

        match_details = None
        search_terms = match_query.split(" vs ")

        # Logic to find matchId and teamIds from live match list
        for category in data.get('typeMatches', []):
            for series in category.get('seriesMatches', []):
                wrapper = series.get('seriesAdWrapper')
                if wrapper and wrapper.get('matches'):
                    try:
                        m_info = wrapper['matches'][0].get('matchInfo', {})
                        m_title = f"{m_info.get('team1', {}).get('teamName')} vs {m_info.get('team2', {}).get('teamName')}"
                        if all(term.strip() in m_title for term in search_terms):
                            match_details = {
                                "id": m_info.get('matchId'),
                                "t1_id": m_info.get('team1', {}).get('teamId'),
                                "t2_id": m_info.get('team2', {}).get('teamId'),
                                "t1_name": m_info.get('team1', {}).get('teamName'),
                                "t2_name": m_info.get('team2', {}).get('teamName')
                            }
                            break
                    except (KeyError, IndexError, TypeError) as e:
                        logger.warning(f"⚠️ Error parsing match info: {e}")
                        continue

        if not match_details:
            logger.warning(f"⚠️ No live match found for '{match_query}'")
            return f"Could not find a live match for '{match_query}'."

        final_output = f"Match Found: {match_details['t1_name']} vs {match_details['t2_name']}\n"

        for t_key in [('t1_id', 't1_name'), ('t2_id', 't2_name')]:
            t_name = match_details[t_key[1]]
            t_id= match_details[t_key[0]]

            squad_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_details['id']}/team/{t_id}"
            s_res = requests.get(squad_url, headers=HEADERS)
            s_data = s_res.json()

            final_output += f"\n--- {t_name} Squad ---\n"
            for group in s_data.get('player', []):
                if group.get('category') == "playing XI":
                    for p in group.get('player', []):
                        final_output += f"- {p.get('name')} ({p.get('role')}) [ID: {p.get('id')}]\n"

        cache.set(cache_key, final_output)
        return final_output

    except requests.RequestException as e:
        return f"Error (Request failed): {str(e)}"
    except (KeyError, ValueError):
        return f"Error: Could not parse match data from API"

2026-02-26 13:21:31,972 - INFO - ✅ LangChain tools imported successfully
2026-02-26 13:21:33,417 - INFO - ✅ Tavily Search imported successfully
2026-02-26 13:21:33,418 - INFO - ✅ Requests library imported successfully
2026-02-26 13:21:33,419 - INFO - ✅ Tavily Search client initialized successfully


## Phase 3: Eco-Scout Agent (Environment-Based Selection)

**Purpose**: Create an AI agent that selects fantasy teams based on environmental factors (weather, pitch, venue).

**Agent Strategy**:
1. Fetch venue details (stadium name, city, pitch history)
2. Fetch weather conditions (temperature, conditions, description)
3. Analyze pitch behavior (batting vs bowling friendly)
4. Retrieve playing squads
5. Select 11 players based on environmental factors ONLY

**Tool Execution Order**: venue → weather → pitch → squads  
**Selection Criteria**: Weather + Pitch conditions (environment-driven, not form-driven)

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from typing import List, Optional
from pydantic import BaseModel, Field

class FantasyPlayer(BaseModel):
    """Information about a single selected fantasy player."""
    player_id: str = Field(description="The unique ID of the player from the squad tool")
    name: str = Field(description="Full name of the player")
    team: str = Field(description="Team name of the player")
    role: str = Field(description="Role: BAT, BOWL, AR, or WK")
    scout_logic: str = Field(description="Detailed reasoning for selecting this player")

# 2. CREATE THIS NEW CONTAINER CLASS
class FantasyTeam(BaseModel):
    """The full squad of 11 players."""
    reasoning_summary: str = Field(description="Overall logic for the whole team")
    players: List[FantasyPlayer] = Field(description="A list of exactly 11 selected players")





In [ ]:
from langchain.agents import create_agent

# Define tool list - same cached tools used by Eco-Scout
eco_agent_tools = [get__weathr, get_match_venue, get_match_environment, get_cricket_squads_by_match]

system_prompt="""Your Role:
You are the "Eco-Scout" Agent — a Cricket Fantasy Analyst whose decisions are STRICTLY based on tool outputs provided during this session.

You do not use memory, prior knowledge, or assumptions outside of tool responses.

You are allowed to reason ONLY on data explicitly returned by tools.

Short basic instruction:
Generate a Generic Fantasy Playing 11 for today's match ONLY AFTER all mandatory tools successfully return usable data.

ABSOLUTE TOOL-FIRST EXECUTION POLICY (NON-NEGOTIABLE):

You MUST attempt to call all mandatory tools in the exact order defined below.

A tool is considered SUCCESSFUL if it returns structured, readable, non-empty data.

You MUST NOT refuse execution merely due to uncertainty about tool availability.

GLOBAL SAFETY RULE (REFINED AND FINAL):

You MUST return the refusal message ONLY IF:

A mandatory tool is explicitly called, AND

That tool explicitly fails, returns null, empty, or unusable output.

If tools return usable data, you MUST proceed.

Refusal message (EXACT TEXT, ONLY IF TRIGGERED):

"I cannot generate a Fantasy Playing 11 because required tools are unavailable. No tool data = no decision."

MANDATORY TOOL EXECUTION ORDER (STRICT):

Step 1 — Venue (MANDATORY TOOL)
Tool Name: get_match_venue

Rules:

Call this tool FIRST.

Do NOT assume teams, venue, or city.

Extract and store venue and city.

If tool explicitly fails or returns null/empty → REFUSE.

Step 2 — Weather (MANDATORY TOOL)
Tool Name: get__weathr

Rules:

Use ONLY the city returned from Step 1.

Use today's date in YYYY-MM-DD format.

Tool must return temperature and conditions.

If tool fails, errors, or returns unusable text → REFUSE.

You are NOT allowed to infer weather without this tool.

Step 3 — Pitch (MANDATORY TOOL)
Tool Name: get_match_environment

Rules:

Use ONLY venue and city from Step 1.

Tool must return pitch behavior (e.g., flat, slow, spin-friendly, pace-friendly).

If pitch data is missing or unclear → REFUSE.

No cricket knowledge may be used without this tool.

Step 4 — Squads (MANDATORY TOOL)
Tool Name: get_cricket_squads_by_match

Rules:

Must return confirmed squads for BOTH teams.

Player roles (BAT / BOWL / AR / WK) must be explicitly provided.

You MUST NOT invent players or roles.

If any squad or role data is missing → REFUSE.

ENVIRONMENTAL REASONING (ONLY AFTER ALL TOOLS SUCCEED):

Create a section titled:
Environmental Reasoning Summary

Rules:

Use ONLY tool-returned data.

Summarize:

Venue characteristics

Weather impact (temperature, conditions)

Pitch behavior

Identify favored PLAYER TYPES (not player names).

NO tool data → NO reasoning.

STRICT SELECTION LOGIC:

Select players ONLY from returned squads.

Selection must be driven ONLY by:

Weather

Pitch behavior

You MUST NOT consider:

Player form

Reputation

Past performance

Popularity

Matchups

Intuition

OUTPUT INSTRUCTIONS:
You MUST provide your final answer by satisfying the provided schema FantasyTeam
- Reasoning Summary: Detail the tool-returned weather and pitch data.
- Playing 11: Select 11 players (1 WK, 4 BOWL, 6 others).
- Use ONLY the player IDs and names provided by the tools.

---
CRITICAL OUTPUT RULE:
1. You MUST NOT provide the final 11 as a text list or markdown table in your response.
2. You MUST provide the final 11 by calling the structured_output tool.
3. Every player MUST include their 'player_id' as returned by the get_cricket_squads_by_match tool.
4. If you do not include the 'player_id', the selection is INVALID.
"""



# Create Eco-Scout Agent with cached tools
eco_agent_executor = create_agent(
    model=model,
    tools=eco_agent_tools,
    system_prompt=system_prompt,# This forces the agent to use the Pydantic schema for its final response
    response_format=ToolStrategy(FantasyTeam)
)

In [ ]:
import logging

logging.basicConfig(level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s |%(message)s",filename='app.log',filemode='a')
logger=logging.getLogger(__name__)




## Phase 4: Form-Scout Agent (Form-Based Selection)

**Purpose**: Create an AI agent that selects fantasy teams based on player form and venue scoring patterns.

**Agent Strategy**:
1. Fetch venue details & scoring patterns (high-scoring vs low-scoring)
2. Retrieve playing squads with player IDs
3. Fetch recent T20 form for all 22 players (batting SR, bowling wickets)
4. Select 11 players based on momentum and venue alignment

**Tool Execution Order**: venue & squads → batch player stats  
**Selection Criteria**: Player form (SR > 140, wickets) + Venue scoring patterns

In [ ]:
from langchain.tools import tool
import requests

# ==========================================
# HELPER FUNCTIONS FOR PLAYER STATS (Form-Scout Only)
# ==========================================

def fetch_single_player_stat(player_id: str) -> dict:
    """Helper function to fetch and parse T20 stats for one player with comprehensive error handling."""
    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"

    try:
        if not HEADERS.get('x-rapidapi-key'):
            logger.error(f"❌ RapidAPI key not configured for player {player_id}")
            return {"id": player_id, "error": "API key not configured"}

        response = requests.get(url, headers=HEADERS, timeout=10)

        # Handle specific HTTP status codes
        if response.status_code == 401:
            logger.error(f"❌ Authentication failed for player {player_id}: 401")
            return {"id": player_id, "error": "API authentication failed"}
        elif response.status_code == 403:
            logger.error(f"❌ Access forbidden for player {player_id}: 403")
            return {"id": player_id, "error": "API access forbidden"}
        elif response.status_code == 404:
            logger.warning(f"⚠️ Player {player_id} not found: 404")
            return {"id": player_id, "error": "Player not found in database"}
        elif response.status_code != 200:
            logger.error(f"❌ API error for player {player_id}: {response.status_code}")
            return {"id": player_id, "error": f"API returned {response.status_code}"}

        response.raise_for_status()  # Raise on 4xx/5xx
        player_data = response.json()

    except requests.Timeout:
        logger.error(f"❌ Timeout fetching player {player_id} (10s)")
        return {"id": player_id, "error": "API request timeout"}
    except requests.ConnectionError as e:
        logger.error(f"❌ Connection error for player {player_id}: {e}")
        return {"id": player_id, "error": f"Connection error: {str(e)}"}
    except requests.RequestException as e:
        logger.error(f"❌ Request error for player {player_id}: {e}")
        return {"id": player_id, "error": f"API request failed: {str(e)}"}
    except ValueError as e:
        logger.error(f"❌ Invalid JSON response for player {player_id}: {e}")
        return {"id": player_id, "error": "Invalid JSON response"}
    except Exception as e:
        logger.error(f"❌ Unexpected error fetching player {player_id}: {e}")
        return {"id": player_id, "error": f"Unexpected error: {str(e)}"}

    # Parse player data with error handling
    t20_batting = []
    t20_bowling = []
    player_name = "Unknown"
    player_role = "Unknown"

    try:
        player_name = player_data.get("name", "Unknown")
        player_role = player_data.get("role", "Unknown")

        # Batting Logic with error handling
        try:
            for row in player_data.get('recentBatting', {}).get('rows', []):
                try:
                    vals = row.get('values', [])
                    if len(vals) > 2 and any(fmt in vals[3] for fmt in ["T20", "T20I"]):
                        score_str = vals[1]
                        if "(" in score_str:
                            raw_runs = score_str.split("(")[0]
                            clean_runs = raw_runs.replace("*", "").strip()
                            runs = int(clean_runs)
                            balls = int(score_str.split("(")[1].replace(")", ""))
                            sr = round((runs / balls) * 100, 2) if balls > 0 else 0
                            t20_batting.append({"runs": runs, "sr": sr})
                except (ValueError, IndexError) as e:
                    logger.warning(f"⚠️ Error parsing batting stats for {player_name} ({player_id}): {e}")
                    continue
        except Exception as e:
            logger.warning(f"⚠️ Error processing batting data for {player_name} ({player_id}): {e}")

        # Bowling Logic with error handling
        try:
            for row in player_data.get('recentBowling', {}).get('rows', []):
                try:
                    vals = row.get('values', [])
                    if len(vals) > 2 and any(fmt in vals[3] for fmt in ["T20", "T20I"]):
                        bowl_str = vals[1]
                        if "-" in bowl_str:
                            wkts = int(bowl_str.split("-")[0])
                            runs = int(bowl_str.split("-")[1])
                            avg = round(runs / wkts, 2) if wkts > 0 else runs
                            t20_bowling.append({"wkts": wkts, "avg": avg})
                except (ValueError, IndexError) as e:
                    logger.warning(f"⚠️ Error parsing bowling stats for {player_name} ({player_id}): {e}")
                    continue
        except Exception as e:
            logger.warning(f"⚠️ Error processing bowling data for {player_name} ({player_id}): {e}")

    except Exception as e:
        logger.error(f"❌ Unexpected error parsing player {player_id} data: {e}")
        return {"id": player_id, "error": f"Data parsing error: {str(e)}"}

    logger.info(f"✅ Player stats parsed: {player_name} ({player_id})")
    return {
        "id": player_id,
        "name": player_name,
        "role": player_role,
        "bat_form": t20_batting,
        "bowl_form": t20_bowling
    }

@tool
def get_batch_player_stats(player_ids: list) -> str:
    """Fetches T20 form for all players with caching. Caches individual players to avoid redundant API calls."""
    batch_results = []
    errors = []
    successful = 0

    try:
        if not player_ids:
            logger.warning("⚠️ No player IDs provided for batch stats fetch")
            return "Error: Empty player ID list provided"

        logger.info(f"📊 Fetching stats for {len(player_ids)} players...")

        for p_id in player_ids:
            try:
                p_key = f"player_{p_id}"

                # Check cache first (no file I/O per player)
                if cache.exists(p_key):
                    try:
                        cached_value = cache.get(p_key)
                        batch_results.append(cached_value['data'])
                        logger.debug(f"✅ Cache hit for player {p_id}")
                        successful += 1
                        continue
                    except Exception as e:
                        logger.warning(f"⚠️ Cache retrieval error for player {p_id}: {e}")

                # Fetch and cache if not available
                player_stats = fetch_single_player_stat(str(p_id))

                # Check for errors in the returned data
                if "error" in player_stats:
                    errors.append(f"Player {p_id}: {player_stats['error']}")
                    logger.warning(f"⚠️ Error for player {p_id}: {player_stats['error']}")
                else:
                    successful += 1

                try:
                    cache.set(p_key, player_stats)  # Auto-saves with timestamp
                except Exception as e:
                    logger.error(f"❌ Failed to cache player {p_id}: {e}")

                batch_results.append(player_stats)

            except Exception as e:
                logger.error(f"❌ Unexpected error processing player {p_id}: {e}")
                errors.append(f"Player {p_id}: {str(e)}")
                batch_results.append({"id": p_id, "error": str(e)})

        logger.info(f"📊 Batch complete: {successful}/{len(player_ids)} successful")

        if errors:
            logger.warning(f"⚠️ {len(errors)} errors encountered: {', '.join(errors[:3])}")

        return str(batch_results)

    except Exception as e:
        logger.error(f"❌ Critical error in batch player stats: {e}")
        return f"Error: Failed to fetch batch player stats: {str(e)}"

# Define Tool List for Form-Scout Agent (includes cached player stats)
form_agent_tools = [get_match_venue, get_cricket_squads_by_match, get_batch_player_stats]

In [ ]:
from langchain.agents import create_agent

system_prompt="""Your Role:
You are the "Form-Scout" Agent. Your decisions are STRICTLY driven by player momentum and venue scoring patterns.

CRITICAL INSTRUCTION: To avoid infinite loops, you MUST follow a strictly sequential workflow.
- Once a tool returns data, you MUST NOT call that same tool again for the same match.
- Use the data already received to proceed to the next step.

Step 1 — Venue & Roster (MANDATORY - ONE-TIME CALL):
1. Call `get_match_venue` ONLY ONCE to get stadium name, city, and scoring history.
   - If you already see "Venue Found" in your history, DO NOT call this tool again.
2. Call `get_cricket_squads_by_match` ONLY ONCE to get the 22 player names and IDs.
   - If you already have the squad list, DO NOT call this tool again. pass the match query like this Eg. Australia vs Ireland

Step 2 — Batch Form Retrieval (MANDATORY):
- Collect ALL 22 Player IDs into a single list.
- Call `get_batch_player_stats` EXACTLY ONCE with the list.
- Do NOT call stats for players individually.

FORM ANALYSIS & SELECTION LOGIC:
1. Identifying Momentum: Use stats from Step 2 (SR > 140 or Wickets).
2. Venue Alignment:
   - High average scores (>170) = Prioritize Power-Hitters.
   - Low average scores (<140) = Prioritize Wicket-Taking Bowlers.
3. Value Picks: Select All-rounders (AR) with both Batting & Bowling stats.

STRICT ROSTER CONSTRAINTS:
- Exactly 11 players.
- Exactly 1 Wicket Keeper (Highest recent T20 runs).
- Exactly 4 Bowlers (Top 4 wicket-takers in recent form).
- Remaining 6: Mix of All-rounders and Batsmen based on Venue.

OUTPUT INSTRUCTIONS:
You MUST provide your final answer by satisfying the provided schema FantasyTeam
- Reasoning Summary: Detail the tool-returned weather and pitch data.
- Playing 11: Select 11 players (1 WK, 4 BOWL, 6 others).
- Use ONLY the player IDs and names provided by the tools.

---
CRITICAL OUTPUT RULE:
1. You MUST NOT provide the final 11 as a text list or markdown table in your response.
2. You MUST provide the final 11 by calling the structured_output tool.
3. Every player MUST include their 'player_id' as returned by the get_cricket_squads_by_match tool.
4. If you do not include the 'player_id', the selection is INVALID.

HARD FAILURE:
If any tool fails or returns empty, output ONLY: "Unable to generate Playing 11 due to missing tool data."
"""

# Create Form-Scout Agent with cached tools (includes player stats caching)
form_agent_executor = create_agent(
    model=model,
    tools=form_agent_tools,
    system_prompt=system_prompt,response_format=ToolStrategy(FantasyTeam)
)

## Phase 5: Match Management & Monitoring

**Purpose**: Fetch today's matches, monitor their status, and trigger agents when matches start.

**Components**:
- `get_todays_match_list()` - Fetch matches from ICC T20 World Cup 2026 series
- `store_daily_matches()` - Smart cache: fetches only if cache is from previous day
- `load_stored_matches()` - Retrieve cached matches
- `check_match_status()` - Poll Cricbuzz API for match state (Preview → In Progress → Result)
- `check_all_matches()` - Group matches into started/upcoming categories

**Scheduler Logic**:
- 7:00 AM: Fetch and store today's matches (one-time)
- Every 5 min: Check which matches have started
- On Match Start: Trigger agent to generate fantasy team

In [ ]:
from datetime import datetime

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/upcoming"

def get_todays_match_list():
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    print(data)

    # Get today's date in 'MMM DD' format (e.g., 'Feb 10')
    today_str = datetime.now().strftime("%b %d")

    matches_today = []
    target_series = "ICC Men's T20 World Cup 2026"

    # Traverse the JSON structure
    for match_type_group in data.get('typeMatches', []):
       for series_group in match_type_group.get('seriesMatches', []):
            wrapper = series_group.get('seriesAdWrapper')

            # FILTER 1: Check if seriesName matches exactly
            if wrapper and wrapper.get('seriesName') == target_series:
                print('inside')
                for match in wrapper.get('matches', []):
                    match_info = match.get('matchInfo', {})
                    status = match_info.get('status', "")
                    print(status)
                    print(today_str)

                    # Check if the match status mentions today's date (Feb 10)
                    if today_str in status:
                        print('inside')
                        team1 = match_info.get('team1', {}).get('teamName')
                        team2 = match_info.get('team2', {}).get('teamName')
                        match_id = match_info.get('matchId')
                        venue = match_info.get('venueInfo', {}).get('ground')

                        matches_today.append({
                            "match": f"{team1} vs {team2}",
                            "id": match_id,
                            "venue": venue,
                            "status": status
                        })

    return matches_today

# # Execute and Print
# todays_games = get_todays_match_list()
# todays_games


In [ ]:
import json
import requests
from datetime import datetime
from pathlib import Path

# ==============================
# Configuration
# ==============================
MATCH_STORAGE_FILE = "daily_matches.json"


# ==============================
# STEP 1 — Fetch and Store Today's Matches
# ==============================
def store_daily_matches():
    """
    Smart Cache: Fetches matches from API ONLY IF the stored file
    is missing or from a previous day.
    """
    try:
        # 1. Check if file exists
        if Path(MATCH_STORAGE_FILE).exists():
            with open(MATCH_STORAGE_FILE, "r") as f:
                storage = json.load(f)

            # 2. Extract stored date (e.g., "2026-02-19")
            stored_date = storage.get("timestamp", "").split("T")[0]
            current_date = datetime.now().isoformat().split("T")[0]

            # 3. If dates match, return the cached matches and skip API
            if stored_date == current_date:
                print(f"📦 Loading matches from cache (Date: {stored_date})")
                return storage.get("matches", [])

        # 4. If no cache or old cache, call the API
        print("🌐 Cache expired or missing. Fetching new matches from API...")
        todays_games = get_todays_match_list()

        # 5. Save new data with fresh timestamp
        storage_data = {
            "timestamp": datetime.now().isoformat(),
            "matches": todays_games
        }

        with open(MATCH_STORAGE_FILE, "w") as f:
            json.dump(storage_data, f, indent=4)

        print(f"✅ API Fetch Successful: Stored {len(todays_games)} matches")
        return todays_games

    except Exception as e:
        print(f"❌ Error in match storage logic: {e}")
        return []


# ==============================
# STEP 2 — Load Stored Matches
# ==============================
def load_stored_matches():
    """Load stored matches"""
    if Path(MATCH_STORAGE_FILE).exists():
        with open(MATCH_STORAGE_FILE, "r") as f:
            data = json.load(f)
            return data.get("matches", [])
    return []


# ==============================
# STEP 3 — Check Match Status
# ==============================
def check_match_status(match_id):
    """
    Checks both 'state' and 'status' to confirm if the match is actually in play.
    'state' is the machine phase (Preview, In Progress, Result).
    'status' is the text description.
    """
    try:
        url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}"
        response = requests.get(url, headers=HEADERS, timeout=10)
        data = response.json()

       # Directly get from the top level
        state = data.get("state", "").lower()
        status = data.get("status", "").lower()


        # Logic: If state is "In Progress", it has definitely started.
        # If state is "Preview", it has NOT started yet.
        match_started = (
            state in ["in progress", "live", "innings break"] or
            any(word in status for word in ["opt to", "elect to", "lead by", "need", "won by"])
        )

        # Safety catch: If it's a "Preview", it definitely hasn't started
        if state == "preview" or state== "complete":
            match_started = False

        return {
            "match_started": match_started,
            "state": state,
            "status": status
        }

    except requests.RequestException as e:
        print(f"❌ Network error checking status: {e}")
        return {"match_started": False, "status": "error"}
    except (KeyError, ValueError) as e:
        print(f"❌ Parse error checking status: {e}")
        return {"match_started": False, "status": "error"}


# ==============================
# STEP 4 — Check All Matches
# ==============================
def check_all_matches():
    """Checks status of all stored matches and returns them grouped by status."""
    matches = load_stored_matches()

    if not matches:
        print("⚠️ No matches stored. Fetch first.")
        return [], []

    print(f"\n🔍 Checking {len(matches)} matches...\n")

    started_matches = []
    upcoming_matches = []

    for match in matches:
        match_id = match.get("id")
        match_name = match.get("match")

        status_info = check_match_status(match_id)

        # Attach the live status and state to the match object
        match["current_status"] = status_info["status"]
        match["current_state"] = status_info["state"]

        if status_info["match_started"]:
            print(f"🎯 MATCH STARTED → {match_name} ({status_info['status']})")
            started_matches.append(match)
        else:
            print(f"⏳ NOT STARTED  → {match_name} ({status_info['status']})")
            upcoming_matches.append(match)

    # Return both lists so you can use them in your main logic
    return started_matches, upcoming_matches

# ==============================
# CACHE CLEANUP ON STARTUP
# ==============================
print(f"📊 Cache Status: {cache.size()} entries")
cache.cleanup(days=7)

2026-02-26 13:21:34,270 - INFO - ℹ️ Cleanup: No expired entries found (threshold: 7 days)


📊 Cache Status: 1 entries


0

## Phase 6: Agent Orchestration & Supervision

**Purpose**: Create a supervisor agent that manages the workflow and delegates to Eco-Scout or Form-Scout.

**Supervisor Agent (Fantasy Cricket Director)**:
- Verifies only LIVE matches can have teams generated
- Routes requests to appropriate scout (Form-Scout or Eco-Scout)
- Combines both scouts' outputs for "Best Overall Team" selection
- Prevents team generation for upcoming (non-started) matches

**Delegation Tools**:
- `get_started_matches_report()` - Check which matches are currently LIVE
- `call_eco_scout(match)` - Invoke Eco-Scout agent
- `call_form_scout(match)` - Invoke Form-Scout agent

**Workflow**:
1. Check if any matches are live
2. Route user request to appropriate agent(s)
3. Return final fantasy team selection

In [ ]:
@tool
def get_started_matches_report() -> str:
    """
    Checks all daily matches and returns a report of matches that
    have ALREADY STARTED (Live, In Progress, or Complete).
    Automatically fetches today's matches if the cache is empty.
    """
    # FIX: Ensure we have today's matches in the cache first!
    # This will either load from cache OR fetch from API if needed.
    matches = store_daily_matches()

    if not matches:
        return "⚠️ Error: Could not retrieve any matches for today. Please check your API connection."

    # Now that we are sure 'matches' is populated, check their status
    started, upcoming = check_all_matches()

    if not started:
        return "No matches have started yet. All scheduled matches are still in 'Upcoming' status."

    report = "🎯 Currently Active/Started Matches:\n"
    for m in started:
        report += f"- {m['match']} (ID: {m['id']})\n"
        report += f"   Status: {m.get('current_status', 'Live')}\n"
        report += f"   Venue: {m['venue']}\n"

    return report
@tool
def call_form_scout(match_details: str) -> str:
    """
    Delegates to the Form-Scout Expert for player momentum and stat analysis.
    Args:
        match_details: The names of the two teams and date (e.g., 'West Indies vs Italy Feb 19').
    """
    # Safety: If the supervisor accidentally passes a dict, extract the string
    if isinstance(match_details, dict):
        match_details = match_details.get("match_details", str(match_details))

    # 1. Format the input EXACTLY like the successful manual stream call
    agent_input = {
        "messages": [
            {
                "role": "user",
                "content": f"Pick the Form-Scout 11 for {match_details}."
            }
        ]
    }

    try:
        # 2. Call invoke using the messages list structure
        response = form_agent_executor.invoke(agent_input)

        # 3. Extract the CONTENT of the AIMessage specifically
        if "messages" in response and len(response["messages"]) > 0:
            final_message = response["messages"][-1]

            # Check if content exists (preventing empty string returns)
            if hasattr(final_message, 'content') and final_message.content:
                return final_message.content

        return "Form-Scout was unable to generate a team. Check if player data is available."

    except Exception as e:
        return f"Error in Form-Scout delegation: {str(e)}"
@tool
def call_eco_scout(match: str) -> str:
    """
    Delegates to the Eco-Scout Expert for environment and pitch analysis.
    Args:
        match: The names of the two teams and date (e.g., 'West Indies vs Italy Feb 19').
    """
    # 1. Format the input to match the working 'messages' schema
    agent_input = {
        "messages": [
            {
                "role": "user",
                "content": f"Pick the Eco-Scout 11 for {match}."
            }
        ]
    }

    try:
        # 2. Invoke the agent with the message list
        response = eco_agent_executor.invoke(agent_input)

        # 3. Extract only the text content from the last AIMessage
        if "messages" in response and len(response["messages"]) > 0:
            final_message = response["messages"][-1]
            return final_message.content

        return "Eco-Scout failed to provide a readable response."

    except Exception as e:
        return f"Error delegating to Eco-Scout: {str(e)}"

In [ ]:
supervisor_tools = [get_started_matches_report, call_form_scout, call_eco_scout]

supervisor_prompt = """Your Role:
You are the "Fantasy Cricket Director" — a strict supervisory manager. You coordinate "Form-Scout" and "Eco-Scout".

CRITICAL: The Scouts return data as a JSON object (containing reasoning_summary and a list of players with player_id). You must ensure the 'player_id' is NEVER lost.

========================================
CASE 0: MATCH LIST ONLY
========================================
If user asks for today's matches:
1. Call 'get_started_matches_report()'.
2. Return the text report exactly. STOP.

========================================
STRICT PROCEDURE FOR TEAM REQUESTS
========================================
STEP 1: Call 'get_started_matches_report()'.
- If "No matches have started yet" → Respond "No live matches today. Team generation unavailable." and STOP.
- If live matches exist → Extract the <EXACT_MATCH_NAME>.

STEP 2: ROUTE BASED ON USER INTENT (3 CASES)

----------------------------------------
CASE 1: Form-Scout Request
----------------------------------------
- Call call_form_scout(match_details="Pick the Form-Scout 11 for <EXACT_MATCH_NAME>").
- You will receive a Pydantic JSON object.
- ACTION: Return that JSON object EXACTLY as it is.
- DO NOT summarize. DO NOT strip the player_id.

----------------------------------------
CASE 2: Eco-Scout Request
----------------------------------------
- Call call_eco_scout(match_details="Pick the Eco-Scout 11 for <EXACT_MATCH_NAME>").
- You will receive a Pydantic JSON object.
- ACTION: Return that JSON object EXACTLY as it is.
- DO NOT summarize. DO NOT strip the player_id.

----------------------------------------
CASE 3: Best Overall Team (Merge)
----------------------------------------
1. Call BOTH 'call_form_scout' AND 'call_eco_scout'.
2. You will receive two JSON objects.
3. Compare the players and logic within these two JSON objects.
4. Select the best 11 players.
5. ACTION: Output a SINGLE JSON object in the SAME schema:
   - "reasoning_summary": Your 3-4 line justification for the merge.
   - "players": The list of 11 players, COPYING 'player_id' EXACTLY from the scout data.

========================================
STRICT RULES
========================================
- You are a DATA MANAGER.
- You MUST NOT generate player names or IDs from memory.
- If the final output does not contain 'player_id' for every player, the task is a FAILURE.
- Always preserve the technical JSON structure returned by the scouts.
."""

# --- 4. Final Supervisor Creation ---

supervisor_agent = create_agent(
    model=model,
    tools=supervisor_tools
)



In [ ]:
# 1. Define your user request (e.g., Form-Scout or Eco-Scout)
user_input = "Pick the Form-Scout 11 for today match"

# 2. Use .stream with stream_mode="updates" to see each agent step
print(f"--- Starting Supervisor Stream for: {user_input} ---\n")

for chunk in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="updates",
):
    for node_name, data in chunk.items():
        new_message = data['messages'][-1]

        print(f"\n--- Step: {node_name} ---")

        # 1. Handle Tool Calls (Supervisor deciding what to do)
        if hasattr(new_message, 'tool_calls') and new_message.tool_calls:
            for tool_call in new_message.tool_calls:
                t_name = tool_call['name']
                t_args = tool_call.get('args', {}) # Get args safely

                print(f"🛠️ Action: Calling Tool [{t_name}]")

                # Print arguments only if they exist
                if t_args:
                    print(f"📝 Arguments: {t_args}")
                else:
                    print("📝 Arguments: None (Status Check)")

        # 2. Handle Tool Outputs (Data coming back to Supervisor)
        elif node_name == "tools":
            print(f"✅ Tool Execution Complete.")
            # Use content safely
            content = getattr(new_message, 'content', "")
            print(f"📄 Data Preview: {content[:150]}...")

        # 3. Final Agent Response
        else:
            if new_message.content:
                print(f"🤖 Final Result:\n{new_message.content}")

--- Starting Supervisor Stream for: Pick the Form-Scout 11 for today match ---



2026-02-26 13:21:39,749 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



--- Step: model ---
🛠️ Action: Calling Tool [get_started_matches_report]
📝 Arguments: None (Status Check)
📦 Loading matches from cache (Date: 2026-02-26)

🔍 Checking 1 matches...

🎯 MATCH STARTED → India vs Zimbabwe (zimbabwe opt to bowl)

--- Step: tools ---
✅ Tool Execution Complete.
📄 Data Preview: 🎯 Currently Active/Started Matches:
- India vs Zimbabwe (ID: 139426)
   Status: zimbabwe opt to bowl
   Venue: MA Chidambaram Stadium
...


2026-02-26 13:21:41,027 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



--- Step: model ---
🛠️ Action: Calling Tool [call_form_scout]
📝 Arguments: {'match_details': 'India vs Zimbabwe'}


...

2026-02-26 13:21:45,523 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-26 13:21:45,976 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-26 13:21:47,250 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"



--- Step: tools ---
✅ Tool Execution Complete.
📄 Data Preview: Error in Form-Scout delegation: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_req...


2026-02-26 13:21:48,251 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-26 13:21:48,311 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-26 13:21:48,313 - INFO - Retrying request to /openai/v1/chat/completions in 8.000000 seconds



--- Step: model ---
🛠️ Action: Calling Tool [call_form_scout]
📝 Arguments: {'match_details': 'India vs Zimbabwe Feb 26'}


2026-02-26 13:21:57,088 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


## Execution Flow: End-to-End

**How the System Works**:

```
User Input: "Pick the Form-Scout 11 for today match"
    ↓
Supervisor Agent (Phase 6)
    ├─ Check live matches via get_started_matches_report()
    ├─ Verify match has actually started
    ├─ Route to Form-Scout
    │
    ├─→ Form-Scout Agent (Phase 4)
    │   ├─ Tool 1: get_match_venue() → Checks cache, or fetches if new
    │   ├─ Tool 2: get_cricket_squads_by_match() → Checks cache, or fetches if new
    │   ├─ Tool 3: get_batch_player_stats() → Checks cache, or fetches individual players
    │   └─ Output: 11-player fantasy team based on form
    │
    └─→ Return final team selection
```

**Key Features**:
- ✅ All tools use shared cache file (cricket_cache.json) - Phase 2
- ✅ Both agents reuse cached data - Phase 4
- ✅ Matches monitored every 5 min - Phase 5
- ✅ Supervisor enforces rules (live match only) - Phase 6
- ✅ Streaming agent output for real-time feedback

In [22]:
import schedule
import time
import json
import pytz
import os
from datetime import datetime

# Configuration
IST = pytz.timezone('Asia/Kolkata')
SELECTIONS_FILE = "agent_selections_log.json"

# --- 1. Notification Helper ---
def show_local_notification(title, report_text):
    """WSL Friendly Notification: Prints a bold block and sounds a terminal bell."""
    print("\a") # Terminal Bell
    print("\n" + "="*50)
    print(f"🔔 {title}")
    print("="*50)
    print(report_text)
    print("="*50 + "\n")

# --- 2. 8:00 PM IST: Nightly Lockdown & Storage ---
def run_nightly_lockdown():
    """Triggers at 8 PM IST. Simulates 5 agents and stores results."""
    now_ist = datetime.now(IST).strftime('%H:%M:%S')
    print(f"🌙 [{now_ist}] Starting Nightly Lockdown...")

    # Define the 5 players/agents as per your requirement
    simulated_players = [
        {"user": "Player_1", "type": "Form-Scout"},
        {"user": "Player_2", "type": "Form-Scout"},
        {"user": "Player_3", "type": "Form-Scout"},
        {"user": "Player_4", "type": "Eco-Scout"},
        {"user": "Player_5", "type": "Eco-Scout"}
    ]

    results_to_store = []

    for p in simulated_players:
        print(f"🔄 Processing {p['user']} via {p['type']}...")

        # We ask the supervisor for the team
        query = f"Pick the {p['type']} 11 for today's match."

        try:
            # Call your existing supervisor agent
            response = supervisor_agent.invoke({"messages": [{"role": "user", "content": query}]})
            team_output = response["messages"][-1].content

            results_to_store.append({
                "date": datetime.now(IST).strftime("%Y-%m-%d"),
                "time": now_ist,
                "user": p['user'],
                "agent_type": p['type'],
                "team": team_output
            })
        except Exception as e:
            print(f"❌ Error generating team for {p['user']}: {e}")

    # Save to file
    if os.path.exists(SELECTIONS_FILE):
        with open(SELECTIONS_FILE, "r") as f:
            existing_data = json.load(f)
    else:
        existing_data = []

    existing_data.extend(results_to_store)

    with open(SELECTIONS_FILE, "w") as f:
        json.dump(existing_data, f, indent=4)

    show_local_notification("NIGHTLY LOCKDOWN COMPLETE", f"Stored 5 agent teams in {SELECTIONS_FILE}")

# --- 3. 3:00 PM IST: Discovery Phase ---
def run_discovery_and_notify():
    now_ist = datetime.now(IST).strftime('%H:%M:%S')
    print(f"🕒 [{now_ist}] Running Discovery Phase...")

    store_daily_matches()

    query = "Give me the available matches report for today."
    response = supervisor_agent.invoke({"messages": [{"role": "user", "content": query}]})
    report_text = response["messages"][-1].content

    show_local_notification("DAILY MATCH DISCOVERY", report_text)

# --- 4. Main Scheduler Loop ---
print(f"📅 Scheduler active. System Time: {datetime.now().strftime('%H:%M:%S')}")
print(f"📍 Targeting 15:00 IST (Discovery) and 20:00 IST (Lockdown)")

while True:
    now_ist = datetime.now(IST)
    current_time_ist = now_ist.strftime("%H:%M")

    # Trigger Discovery at 3:00 PM (15:00)
    if current_time_ist == "15:00":
        run_discovery_and_notify()
        time.sleep(61)

    # Trigger Lockdown at 8:00 PM (20:00)
    if current_time_ist == "20:20":
        run_nightly_lockdown()
        time.sleep(61)

    # Heartbeat every minute
    if now_ist.second == 0:
        print(f"💓 Heartbeat: {current_time_ist} IST")

    time.sleep(1)

📅 Scheduler active. System Time: 14:48:07
📍 Targeting 15:00 IST (Discovery) and 20:00 IST (Lockdown)
💓 Heartbeat: 20:19 IST
🌙 [20:20:00] Starting Nightly Lockdown...
🔄 Processing Player_1 via Form-Scout...


2026-02-20 14:50:02,367 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



🔍 Checking 1 matches...

🎯 MATCH STARTED → Oman vs Australia (australia opt to bowl-australia opt to bowl)


2026-02-20 14:50:03,645 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:04,078 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:05,409 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:06,723 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:07,369 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:08,107 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:09,412 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:12,033 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:50:12,372 - INFO - HTTP Request: POST http

🔄 Processing Player_2 via Form-Scout...


...

2026-02-20 14:59:32,764 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-20 14:59:32,766 - INFO - Retrying request to /openai/v1/chat/completions in 8.000000 seconds


🎯 MATCH STARTED → Oman vs Australia (innings break-australia opt to bowl)


2026-02-20 14:59:41,617 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:59:41,716 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-20 14:59:41,717 - INFO - Retrying request to /openai/v1/chat/completions in 4.000000 seconds
2026-02-20 14:59:46,187 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:59:46,281 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-20 14:59:46,282 - INFO - Retrying request to /openai/v1/chat/completions in 6.000000 seconds
2026-02-20 14:59:52,906 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 14:59:52,990 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-20 14:59:52,992 - INFO - Retrying req



🔔 NIGHTLY LOCKDOWN COMPLETE
Stored 5 agent teams in agent_selections_log.json

💓 Heartbeat: 20:20 IST
💓 Heartbeat: 20:32 IST


KeyboardInterrupt: 

In [10]:
nums="111"
result=0

for i in range(len(nums)):

    result=result*2+int(nums[i])


count=0
while result >1:

    if result %2==0:
        result=result//2

    else:
        result=result+1

    count+=1

print(count)

4
